In [16]:
import polars as pl
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from sklearn.decomposition import PCA

df = pl.read_parquet('../../data/processed/player_profiles.parquet')
print(df.shape)
print(df.head())

(9037, 87)
shape: (5, 87)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ player_id ┆ player_na ┆ total_pas ┆ successfu ┆ … ┆ headed_cl ┆ left_foot ┆ right_foo ┆ other_cl │
│ ---       ┆ me        ┆ ses       ┆ l_passes  ┆   ┆ earances  ┆ _clearanc ┆ t_clearan ┆ earances │
│ i64       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ es        ┆ ces       ┆ ---      │
│           ┆ str       ┆ i64       ┆ i64       ┆   ┆ i64       ┆ ---       ┆ ---       ┆ i64      │
│           ┆           ┆           ┆           ┆   ┆           ┆ i64       ┆ i64       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 398833    ┆ Fritz     ┆ 15        ┆ 12        ┆ … ┆ 0         ┆ 0         ┆ 0         ┆ 0        │
│           ┆ Walter    ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ 398831    ┆ Maurizio  ┆ 16        ┆ 11        ┆ … ┆ 0         ┆

In [5]:
numeric_df = df.drop(['player_id', 'player_name', 'id', 'name'])

print(f"Numeric columns: {numeric_df.shape[1]}")
print(numeric_df.columns)

Numeric columns: 83
['total_passes', 'successful_passes', 'avg_pass_length', 'avg_pass_angle', 'passes_under_pressure', 'successful_passes_under_pressure', 'pass_completion_under_pressure_pct', 'total_carries', 'avg_carry_distance', 'progressive_carries', 'avg_carry_duration', 'carries_under_pressure', 'carries_under_pressure_pct', 'total_pressures', 'avg_pressure_duration', 'avg_pressure_loc_x', 'avg_pressure_loc_y', 'total_interceptions', 'successful_interceptions', 'avg_interception_loc_x', 'avg_interception_loc_y', 'interception_success_pct', 'total_recovery', 'avg_recovery_loc_x', 'avg_recovery_loc_y', 'total_duels', 'duels_with_outcome', 'duels_won', 'duels_lost', 'duel_win_pct', 'aerial_duels', 'tackles', 'avg_duel_loc_x', 'avg_duel_loc_y', 'total_fouls_won', 'penalties_won', 'penalty_won_rate', 'defensive_fouls_won', 'free_kicks_won', 'avg_foul_won_loc_x', 'avg_foul_won_loc_y', 'total_fouls_committed', 'penalties_conceded', 'yellow_cards', 'red_cards', 'offensive_fouls', 'secon

In [6]:
# Convert to numpy array for sklearn
X = numeric_df.to_numpy()

# Standardise
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Shape after scaling: {X_scaled.shape}")
print(f"Mean of first column: {X_scaled[:, 0].mean():.4f}")
print(f"Std of first column: {X_scaled[:, 0].std():.4f}")

Shape after scaling: (9037, 83)
Mean of first column: 0.0000
Std of first column: 1.0000


In [7]:
# First run PCA with all components to see variance explained
pca = PCA()
pca.fit(X_scaled)

# Calculate cumulative variance explained
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

# Find how many components explain 95% of variance
n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1

print(f"Components needed for 95% variance: {n_components_95}")
print(f"Variance explained by {n_components_95} components: {cumulative_variance[n_components_95-1]:.4f}")

Components needed for 95% variance: 53
Variance explained by 53 components: 0.9521


In [ ]:
# Check lower threshold
for threshold in [0.80, 0.85, 0.90, 0.95]:
    n = np.argmax(cumulative_variance >= threshold) + 1
    print(f"{threshold*100:.0f}% variance: {n} components")

80% variance: 31 components
85% variance: 37 components
90% variance: 44 components
95% variance: 53 components


In [11]:
# Run PCA with 53 components
pca = PCA(n_components=53)
X_pca = pca.fit_transform(X_scaled)

print(f"Shape before PCA: {X_scaled.shape}")
print(f"Shape after PCA: {X_pca.shape}")


Shape before PCA: (9037, 83)
Shape after PCA: (9037, 53)


In [12]:
component_loadings = pca.components_[0]
column_names = numeric_df.columns

# Sort by absolute value
sorted_indices = np.argsort(np.abs(component_loadings))[::-1]

print("Top 10 metrics in Component 1:")
for i in sorted_indices[:10]:
    print(f"{column_names[i]}: {component_loadings[i]:.4f}")

Top 10 metrics in Component 1:
dispossessed_under_pressure_pct: 0.1853
avg_dispossessed_loc_x: 0.1589
total_dispossessed: 0.1581
dispossessed_under_pressure: 0.1581
total_duels: 0.1581
avg_foul_won_loc_x: 0.1558
avg_foul_committed_loc_x: 0.1542
clearances_under_pressure_pct: 0.1539
total_shots: 0.1533
avg_miscontrol_loc_x: 0.1519


In [13]:
for comp_num in range(3):
    component_loadings = pca.components_[comp_num]
    sorted_indices = np.argsort(np.abs(component_loadings))[::-1]

    print(f"\nTop 5 metrics in Component {comp_num + 1}:")
    for i in sorted_indices[:5]:
        print(f"  {column_names[i]}: {component_loadings[i]:.4f}")


Top 5 metrics in Component 1:
  dispossessed_under_pressure_pct: 0.1853
  avg_dispossessed_loc_x: 0.1589
  total_dispossessed: 0.1581
  dispossessed_under_pressure: 0.1581
  total_duels: 0.1581

Top 5 metrics in Component 2:
  clearances_under_pressure: 0.2572
  total_clearances: 0.2572
  avg_pressure_loc_x: -0.2149
  headed_clearances: 0.2079
  total_passes: 0.1924

Top 5 metrics in Component 3:
  total_carries: 0.2894
  total_passes: 0.2729
  successful_passes: 0.2606
  successful_passes_under_pressure: 0.2471
  avg_pressure_loc_y: -0.2398


In [14]:
# Create a DataFrame with player IDs and their PCA components
player_vectors = pl.DataFrame({
    'player_id': df['player_id'],
    'player_name': df['player_name'],
    **{f'component_{i+1}': X_pca[:, i] for i in range(53)}
})

print(player_vectors.shape)
print(player_vectors.head())

(9037, 55)
shape: (5, 55)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ player_id ┆ player_na ┆ component ┆ component ┆ … ┆ component ┆ component ┆ component ┆ componen │
│ ---       ┆ me        ┆ _1        ┆ _2        ┆   ┆ _50       ┆ _51       ┆ _52       ┆ t_53     │
│ i64       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆ str       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 398833    ┆ Fritz     ┆ 1.750511  ┆ -3.768301 ┆ … ┆ -0.184079 ┆ -0.149358 ┆ 0.073571  ┆ 0.618859 │
│           ┆ Walter    ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ 398831    ┆ Maurizio  ┆ 4.664145  ┆ -4.220592 ┆ … ┆ 0.210781  ┆ -0.50313  ┆ -0.073082 ┆ -0.50440 │
│           ┆ Gaudino   ┆           ┆           ┆   ┆           ┆

In [15]:
player_vectors.write_parquet('../../data/processed/player_vectors.parquet')
print("Player vectors saved successfully")

Player vectors saved successfully


Cosine Similarity


In [17]:
# Get just the component columns as numpy array
vectors = player_vectors.select([f'component_{i+1}' for i in range(53)]).to_numpy()

print(f"Vector matrix shape: {vectors.shape}")

Vector matrix shape: (9037, 53)


In [19]:
def find_similar_players(player_name: str, top_n: int = 10):
    # Find the player's index
    player_names = player_vectors['player_name'].to_list()

    if player_name not in player_names:
        print(f"Player {player_name} not found")
        return

    idx = player_names.index(player_name)

    # Get the player's vector
    player_vector = vectors[idx].reshape(1, -1)

    # Calculate cosine similarity against all players
    similarities = cosine_similarity(player_vector, vectors)[0]

    # Get top N most similar players (excluding the player themselves)
    similar_indices = np.argsort(similarities)[::-1][1:top_n+1]

    print(f"\nPlayers most similar to {player_name}:")
    for i in similar_indices:
        print(f"{player_vectors['player_name'][int(i)]}: {similarities[i]:.4f}")

# Test it
find_similar_players("Lionel Andrés Messi Cuccittini")


Players most similar to Lionel Andrés Messi Cuccittini:
Arjen Robben: 0.8431
Jennifer Hermoso Fuentes: 0.8349
María Francesca Caldentey Oliver: 0.8229
Samantha May Kerr: 0.7905
Edvaldo Izidio Neto: 0.7905
Keita Baldé Diao: 0.7747
Philippe Coutinho Correia: 0.7709
Gabrielle Onguene: 0.7708
Rodrygo Silva de Goes: 0.7687
Édson Arantes do Nascimento: 0.7650
